# RNN Quest

Виконав: Роман, група ШІ-32 (roman.mydzhyn@example.com)

## Про що ця лаба

**Рекурентна нейронна мережа (RNN)** обробляє послідовність по одному елементу й тримає в голові «пам'ять» про все, що бачила раніше. Ця пам'ять називається **прихованим станом** `h`. На кожному кроці мережа бере новий символ і старий стан і рахує з них новий стан. З нього вона передбачає, який символ буде далі.

Тут використовується найпростіший варіант, **Elman RNN**. У неї є лише одна формула оновлення стану, без гейтів, як у LSTM чи GRU.

**Суть квесту:** хтось натренував маленьку RNN так, що вона «завчила» одне конкретне повідомлення. Мені дали тільки її ваги (числа в матрицях), а самого тексту не дали. Щоб прочитати повідомлення, я вручну написав forward pass мережі, тобто прогнав її крок за кроком. Тоді вона сама «надиктувала» завчений текст.

---
Спершу ставлю пакет курсу `lpnlp`. Через нього я отримую завдання від сервера й відправляю відповідь.

In [1]:
!pip install --quiet --ignore-installed https://static.lp-nlp.com/pypi/lpnlp-2025.10.14-py3-none-any.whl requests==2.32.4

Реєструюся на сервері курсу. `lpnlp.start(...)` повідомляє, хто виконує лабу і яку саме (`quest_rnn`). У відповідь сервер дає мені об'єкт `lab`: у ньому є посилання на моє персональне завдання (`lab.payload`), і через нього я потім здаю відповідь (`lab.answer`).

In [2]:
import lpnlp

lab = lpnlp.start(
    email="roman.mydzhyn@example.com",
    lab="quest_rnn"
)

Ваше завдання: https://static.lp-nlp.com/quest-rnn/d7071db0908e.zip. Удачі! █


## Умова завдання

Сервер видає архів з вагами мережі та README. У README описано, як саме працює мережа:

```
x_t = embedding[message[t - 1]]
h_t = tanh(W_h @ x_t + W_h_bias + U_h @ h_{t-1})
y_t = W_y @ h_t
message[t] = vocab_lookup(argmax(y_t))
```

Як я розумію кожен рядок:

1. **`x_t = embedding[...]`**: беру попередній символ і перетворюю його на вектор-ембедінг. Ембедінг — це рядок матриці `E`, по одному рядку на кожен символ словника.
2. **`h_t = tanh(...)`**: новий прихований стан. Він залежить від двох речей:
   * `W_h @ x_t` — що мережа бачить **зараз** (поточний символ);
   * `U_h @ h_{t-1}` — що вона **пам'ятає** з попередніх кроків.

   До суми додається зсув (bias), а `tanh` стискає все в діапазон від −1 до 1, щоб числа не «вибухали».
3. **`y_t = W_y @ h_t`**: зі стану отримую **logits**, по одному числу на кожен символ словника. Чим більше число, тим імовірніше, що цей символ іде наступним.
4. **`argmax`**: беру символ з найбільшим числом. Це **жадібний (greedy) декодинг**, такий самий, як у моїй лабі з генерації тексту, тільки там була GPT-2, а тут RNN.

Спецтокени: `[` означає початок повідомлення (з нього я стартую), `]` означає кінець (на ньому зупиняюсь).

Завантажую архів із завданням. Посилання лежить у `lab.payload["package_url"]`, у кожного студента воно своє. `wget` скачує архів, `unzip` розпаковує його в папку `/content/quest`, а `ls` показує, які файли всередині: ваги матриць у JSON і словник символів.

In [3]:
package_url = lab.payload["package_url"]
print(package_url)
!wget -q {package_url} -O /content/quest.zip && unzip -o -q /content/quest.zip -d /content/quest
!ls -la /content/quest

https://static.lp-nlp.com/quest-rnn/d7071db0908e.zip
total 712
drwxr-xr-x 2 root root   4096 Nov 23 09:02 .
drwxr-xr-x 1 root root   4096 Nov 23 09:02 ..
-rw-rw-r-- 1 root root 130076 Oct 26  2025 embedding.weight.json
-rw-rw-r-- 1 root root    548 Oct 26  2025 quest.bin
-rw-rw-r-- 1 root root   3419 Oct 26  2025 README
-rw-rw-r-- 1 root root 215092 Oct 26  2025 U_h.weight.json
-rw-rw-r-- 1 root root    977 Oct 26  2025 vocab.json
-rw-rw-r-- 1 root root   1349 Oct 26  2025 W_h.bias.json
-rw-rw-r-- 1 root root 172110 Oct 26  2025 W_h.weight.json
-rw-rw-r-- 1 root root 178653 Oct 26  2025 W_y.weight.json


## Завантажую ваги мережі

Кожен файл JSON — це одна матриця, яку я перетворюю на масив `numpy`:

| Змінна | Що це | Розмір |
|---|---|---|
| `vocab` | словник: список усіх символів, які знає мережа | — |
| `E` | ембедінги, по одному вектору на кожен символ | (розмір словника × розмір ембедінгу) |
| `W_h` | ваги «що бачу зараз»: вхід → стан | (розмір стану × розмір ембедінгу) |
| `b_h` | зсув (bias) для стану | (розмір стану) |
| `U_h` | ваги «що пам'ятаю»: попередній стан → новий стан | (розмір стану × розмір стану) |
| `W_y` | вихідний шар: стан → logits для кожного символу | (розмір словника × розмір стану) |

Ваги збережені у форматі PyTorch `nn.Linear`, де матриця має вигляд (вихід × вхід). Тому я множу як `W @ x`, без транспонування. Розміри друкую, щоб переконатися, що всі множення матриць зійдуться.

In [4]:
import json
import numpy as np

D = "/content/quest/"
def load(name):
    return np.array(json.load(open(D + name + ".json")))

vocab = json.load(open(D + "vocab.json"))
E   = load("embedding.weight")   # (vocab, emb)
W_h = load("W_h.weight")         # (hidden, emb)  -- формат torch.nn.Linear
b_h = load("W_h.bias")           # (hidden,)
U_h = load("U_h.weight")         # (hidden, hidden)
W_y = load("W_y.weight")         # (vocab, hidden)
print(E.shape, W_h.shape, b_h.shape, U_h.shape, W_y.shape, len(vocab))

(133, 128) (160, 128) (160,) (160, 160) (133, 160) 133


## Forward pass і декодинг: головна частина

Моя функція `rnn_greedy_decode` по кроках реалізує формули з умови:

1. **Старт.** Прихований стан `h` заповнюю нулями, бо мережа ще нічого не бачила. Перший вхідний символ — це `[`.
2. **Цикл** (максимум 1000 кроків, щоб не зависнути, якщо `]` так і не з'явиться):
   * `x = E[token]` — беру ембедінг поточного символу;
   * `h = np.tanh(W_h @ x + b_h + U_h @ h)` — оновлюю пам'ять, поєднуючи поточний символ і те, що було раніше;
   * `y = W_y @ h` — отримую оцінку для кожного символу словника;
   * `token = argmax(y)` — обираю найімовірніший символ, і він стає входом на наступному кроці;
   * якщо це `]`, повідомлення закінчилося і я виходжу з циклу, інакше дописую символ у результат.
3. Склеюю всі символи в рядок і друкую.

**Чому це працює.** Мережу перенавчили на одному тексті, тож вона його «завчила». Якщо щоразу брати найімовірніший символ, вона відтворює саме цей текст, символ за символом. По суті, прихований стан `h` — це й є «курсор», що пам'ятає, на якому місці повідомлення ми зараз.

**Зв'язок з теорією.** Це той самий цикл авторегресійної генерації, що й у GPT: вихід кроку t стає входом кроку t+1. Різниця лише в тому, як рахується наступний токен: тут проста рекурентна формула, а в GPT трансформер з attention. Слабке місце RNN у тому, що вся пам'ять стиснута в один вектор `h`, і градієнти на довгих послідовностях затухають. Тому для довгих текстів придумали LSTM і GRU, а згодом трансформери.

In [5]:
def rnn_greedy_decode(max_len=1000):
    """Forward pass Elman RNN + жадібний декодинг."""
    h = np.zeros(U_h.shape[0])
    token = vocab.index("[")              # токен початку послідовності
    message = []
    for _ in range(max_len):
        x = E[token]                                  # x_t = embedding[message[t-1]]
        h = np.tanh(W_h @ x + b_h + U_h @ h)          # h_t
        y = W_y @ h                                   # y_t (логіти)
        token = int(np.argmax(y))                     # жадібний вибір
        if vocab[token] == "]":                       # кінець послідовності
            break
        message.append(vocab[token])
    return "".join(message)

message = rnn_greedy_decode()
print(message)

Назва виконавця: https://www.youtube.com/watch?v=dtSmUF0vMKE


## Результат

Мережа «надиктувала» мені повідомлення: `Назва виконавця: https://www.youtube.com/watch?v=dtSmUF0vMKE`.

Тобто це загадка з двох кроків: спершу розкодувати ваги, потім перейти за посиланням. Я перейшов і побачив пісню **Gordosti** гурту **Ragapop**. Отже, моя відповідь на квест — назва виконавця: **Ragapop**.

## Здаю відповідь

`lab.answer(...)` відправляє мою відповідь на сервер курсу, і той перевіряє її. Відповідь **Ragapop** прийнято ✅

In [6]:
lab.answer("Ragapop")

Відповідь правильна ✅
You did it! 🚀 Next step: https://tally.so/r/wMXM0p
